In [10]:
import numpy as np
from LanzaModels import TVL2_1D
from ADMMsRustici import FigueredoSolver
from signalClass import *
import time

In [11]:
np.random.seed(24102000)
n = 512

construct blur matrix

In [12]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [13]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 50)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [14]:
mu = 2
VarModel = TVL2_1D.TVL2_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [15]:
#begin solver construction
np.random.seed(24102002)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = FigueredoSolver.FigueredoSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [16]:
iters = 1500

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [17]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 1500
2 / 1500
3 / 1500
4 / 1500
5 / 1500
6 / 1500
7 / 1500
8 / 1500
9 / 1500
10 / 1500
11 / 1500
12 / 1500
13 / 1500
14 / 1500
15 / 1500
16 / 1500
17 / 1500
18 / 1500
19 / 1500
20 / 1500
21 / 1500
22 / 1500
23 / 1500
24 / 1500
25 / 1500
26 / 1500
27 / 1500
28 / 1500
29 / 1500
30 / 1500
31 / 1500
32 / 1500
33 / 1500
34 / 1500
35 / 1500
36 / 1500
37 / 1500
38 / 1500
39 / 1500
40 / 1500
41 / 1500
42 / 1500
43 / 1500
44 / 1500
45 / 1500
46 / 1500
47 / 1500
48 / 1500
49 / 1500
50 / 1500
51 / 1500
52 / 1500
53 / 1500
54 / 1500
55 / 1500
56 / 1500
57 / 1500
58 / 1500
59 / 1500
60 / 1500
61 / 1500
62 / 1500
63 / 1500
64 / 1500
65 / 1500
66 / 1500
67 / 1500
68 / 1500
69 / 1500
70 / 1500
71 / 1500
72 / 1500
73 / 1500
74 / 1500
75 / 1500
76 / 1500
77 / 1500
78 / 1500
79 / 1500
80 / 1500
81 / 1500
82 / 1500
83 / 1500
84 / 1500
85 / 1500
86 / 1500
87 / 1500
88 / 1500
89 / 1500
90 / 1500
91 / 1500
92 / 1500
93 / 1500
94 / 1500
95 / 1500
96 / 1500
97 / 1500
98 / 1500
99 / 1500
100 / 1500
101 / 15

In [18]:
np.savez_compressed(
    "./FigueredoADMMTVL2-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)